# Static Semantic Search using Word2Vec

## Average Word Embeddings for Research-Paper Abstracts

**Developer Role:** Word2Vec Developer (Static Semantic Search)

---

### Overview

This notebook implements **static semantic representation** for research-paper search using **Word2Vec** (`gensim`).

Word2Vec learns **one fixed vector per word**, independent of the sentence the word appears in — hence *static*. Words that appear in similar contexts end up close together in the vector space, so a query can match an abstract even when they share **no literal vocabulary** (`"car"` ↔ `"automobile"`). This is the key advantage over lexical TF-IDF search.

The trade-off, and the reason this is only the middle arm of the case study: because each word has exactly one vector, *"bank"* (river) and *"bank"* (finance) collapse into the same representation. Contextual models (BERT/SPECTER) resolve that; Word2Vec cannot.

### Document strategy: Average Word Embedding

Word2Vec produces **word** vectors, not **document** vectors. To represent a whole abstract we take the **element-wise mean** of the vectors of all its preprocessed words:

$$\vec{d} = \frac{1}{|T_d|} \sum_{w \in T_d} \vec{w}$$

### Pipeline

```
Abstracts / Query
       ↓
Preprocessing  (lowercase → tokenize → remove punctuation & stop words → lemmatize)
       ↓
Word2Vec  (trained on the corpus, or a pre-trained model)
       ↓
Average of word vectors  →  Embedding vectors (NumPy arrays)
```

### Contents

| Cell | Purpose |
|------|---------|
| 1–3 | Dependencies, imports, NLTK data |
| 4 | The `Word2VecSearcher` class (the deliverable) |
| 5 | Load `abstract_sentences.csv` |
| 6 | Preprocessing walk-through (before → after) |
| 7–9 | Train the model, embed the corpus, embed a query |
| 10 | Inspect the embeddings |
| 11–13 | **Demo run:** semantic search over the dataset |

> **Note:** cells 1–10 are the actual deliverable — embedding generation only. The cosine-similarity/ranking code in cells 11–13 is a **demonstration** that the embeddings work; the ranking component is owned by a separate part of the case study and is deliberately kept out of the `Word2VecSearcher` class.


---
## Cell 1 — Install Dependencies

| Library | Purpose |
|---------|---------|
| `gensim` | The Word2Vec implementation (training from scratch + pre-trained model downloads) |
| `nltk` | Tokenization, the English stop-word list, and the WordNet lemmatizer |
| `numpy` | Stores the embeddings as efficient numerical arrays |
| `pandas` | Loads the research-paper dataset from CSV |

> **Python version:** `gensim` currently ships wheels for **Python ≤ 3.12**. On a newer interpreter, create an environment first (e.g. `uv venv --python 3.12 venv`). Google Colab works out of the box.


In [1]:
# ============================================================
# Cell 1 — Install Dependencies (run once in terminal)
# ============================================================
# pip install gensim nltk numpy pandas
#
# gensim : Word2Vec implementation (training + pre-trained models)
# nltk   : tokenization, English stop-word list, WordNet lemmatizer
# numpy  : stores the embeddings as numerical arrays
# pandas : loads the dataset from CSV (used by the caller/notebook)
#
# NOTE: gensim currently ships wheels for Python <= 3.12. On newer
#       interpreters create an environment first, e.g.:
#           uv venv --python 3.12 venv
# ============================================================

---
## Cell 2 — Imports

| Import | Purpose |
|--------|---------|
| `re`, `string` | Built-ins for basic cleaning (HTML/URLs) and the punctuation set |
| `numpy` | Stores embedding matrices/vectors |
| `word_tokenize` | Splits a sentence into a list of word tokens |
| `stopwords` | The standard English stop-word list ("the", "of", "and", …) |
| `WordNetLemmatizer`, `wordnet` | Reduces inflected words to their base form |
| `Word2Vec` | Trains the model on our corpus |
| `gensim.downloader` | Loads pre-trained vectors such as `word2vec-google-news-300` |


In [2]:
# ============================================================
# Cell 2 — Imports
# ============================================================

import re                                        # Built-in: basic text cleaning (HTML tags, URLs, whitespace)
import string                                    # Built-in: the punctuation character set

import numpy as np                               # Numerical arrays for storing embeddings

import nltk                                      # Classical NLP toolkit (tokenizer, stop words, lemmatizer)
from nltk.corpus import stopwords, wordnet       # English stop-word list + WordNet POS constants
from nltk.stem import WordNetLemmatizer          # Rule/lexicon-based lemmatizer
from nltk.tokenize import word_tokenize          # Splits a sentence into a list of word tokens

from gensim.models import Word2Vec               # Trains a Word2Vec model from scratch on our corpus
import gensim.downloader as api                  # Downloads pre-trained vectors (e.g. word2vec-google-news-300)

---
## Cell 3 — NLTK Resource Bootstrap

NLTK ships the *code*, but the *data* files (tokenizer tables, stop-word list, WordNet, POS tagger) are downloaded separately. This helper fetches anything missing and is a no-op once the files are on disk — so the notebook runs on a fresh machine or a fresh Colab runtime without manual setup.


In [3]:
# ============================================================
# Cell 3 — NLTK Resource Bootstrap
# ============================================================
# NLTK ships code, but the data files (tokenizer models, stop-word
# list, WordNet) must be downloaded once. This helper downloads any
# missing resource quietly so the class works on a fresh machine.
# ============================================================

# (resource path used by nltk.data.find, package name used by nltk.download)
_NLTK_RESOURCES = [
    ("tokenizers/punkt_tab", "punkt_tab"),                                  # sentence/word tokenizer tables
    ("tokenizers/punkt", "punkt"),                                          # tokenizer (older NLTK versions)
    ("corpora/stopwords", "stopwords"),                                     # English stop-word list
    ("corpora/wordnet", "wordnet"),                                         # lexical database used for lemmatization
    ("corpora/omw-1.4", "omw-1.4"),                                         # WordNet multilingual data (WordNet dependency)
    ("taggers/averaged_perceptron_tagger_eng", "averaged_perceptron_tagger_eng"),  # POS tagger for POS-aware lemmatization
]


def ensure_nltk_resources():
    """
    Download the NLTK data files required for preprocessing, if missing.

    Safe to call repeatedly: a resource that is already present on disk
    is detected by `nltk.data.find` and is not downloaded again.
    """
    for resource_path, package_name in _NLTK_RESOURCES:
        try:
            nltk.data.find(resource_path)
        except LookupError:
            nltk.download(package_name, quiet=True)

In [4]:
# Fetch the NLTK data now so the preprocessing cells below run cleanly
ensure_nltk_resources()

print("NLTK resources ready!")
print(f"English stop words available: {len(stopwords.words('english'))}")

NLTK resources ready!
English stop words available: 198


---
## Cell 4 — Define the `Word2VecSearcher` Class

This is the deliverable. Three things to note:

1. **`preprocess()`** performs the full classical NLP pipeline — lowercase → tokenize → remove punctuation, numbers and stop words → POS-aware lemmatize. Unlike the BERT arm (where preprocessing would *destroy* the context the transformer needs), Word2Vec **requires** it: the model learns from raw token co-occurrence, so noise tokens and inflected duplicates directly degrade the vectors.
2. **`_average_word_vectors()`** implements the Average Word Embedding strategy, skipping out-of-vocabulary words and falling back to a zero vector when nothing is known.
3. **The same `preprocess()` runs on both abstracts and queries**, which is what guarantees the query vector lands in the same space as the document vectors.

The constructor can either **train from scratch** on the corpus (default — fast, offline, domain-specific vocabulary) or **load pre-trained vectors** with `use_pretrained=True`.


In [5]:
# ============================================================
# Cell 4 — Define the Word2VecSearcher Class
# ============================================================

class Word2VecSearcher:
    """
    Static Semantic Search using Word2Vec + Average Word Embeddings.

    Unlike TF-IDF (which matches literal words) this class represents
    text by MEANING: words that appear in similar contexts receive
    similar vectors, so a query can match an abstract even when they
    share no vocabulary ("car" vs "automobile").

    Unlike BERT/SPECTER, the vectors are STATIC: each word has exactly
    one vector regardless of context, so "bank" (river) and "bank"
    (finance) collapse into a single representation. That trade-off is
    precisely what this arm of the case study is meant to measure.

    Document strategy
    -----------------
    Word2Vec produces WORD vectors, not document vectors. This class
    uses the Average Word Embedding approach: the document vector is
    the element-wise mean of the vectors of all its in-vocabulary
    preprocessed tokens.

    Parameters
    ----------
    abstracts : list of str
        The research-paper abstracts to embed.
    use_pretrained : bool, default False
        False -> train a Word2Vec model from scratch on `abstracts`
                 (fast, offline, and the vocabulary matches the
                 scientific domain of the corpus).
        True  -> load pre-trained vectors via gensim-data
                 (broader general-English vocabulary, but a large
                 one-off download: ~1.6 GB for Google News).
    pretrained_model_name : str, default "word2vec-google-news-300"
        gensim-data model name, used only when `use_pretrained=True`.
    vector_size : int, default 300
        Embedding dimensionality when training from scratch.
        (Ignored for pre-trained models, which fix their own size.)
    window : int, default 5
        Context window size used during training.
    min_count : int, default 2
        Words occurring fewer than `min_count` times are ignored.
    epochs : int, default 30
        Training passes over the corpus. Our corpus is small
        (a few hundred abstracts), so more epochs help.
    sg : int, default 1
        1 = skip-gram (better for small corpora and rare technical
        terms), 0 = CBOW (faster).
    workers : int, default 4
        Worker threads used for training.
    random_seed : int, default 42
        Seed for reproducible training runs.

    Attributes
    ----------
    tokenized_corpus : list of list of str
        The preprocessed token lists, one per abstract.
    word_vectors : gensim.models.KeyedVectors
        The lookup table mapping a word to its static vector.
    vector_size : int
        Dimensionality of every embedding produced by this class.

    Methods
    -------
    preprocess(text)
        Lowercase → tokenize → strip punctuation/stop words → lemmatize.
    get_corpus_embeddings()
        Returns a 2D NumPy array of shape (n_papers, vector_size).
    get_query_embedding(query_string)
        Returns a 1D NumPy array of shape (vector_size,).
    """

    def __init__(
        self,
        abstracts,
        use_pretrained=False,
        pretrained_model_name="word2vec-google-news-300",
        vector_size=300,
        window=5,
        min_count=2,
        epochs=30,
        sg=1,
        workers=4,
        random_seed=42,
    ):
        """Preprocess the corpus and build (train or load) the Word2Vec model."""
        # ---- Store the raw corpus -------------------------------------
        self.abstracts = abstracts

        # ---- Set up the preprocessing tools ---------------------------
        # Downloaded once, then reused for every abstract and every query.
        ensure_nltk_resources()
        self.stop_words = set(stopwords.words("english"))   # e.g. "the", "of", "and"
        self.lemmatizer = WordNetLemmatizer()               # "networks" -> "network"
        self.punctuation = set(string.punctuation)          # ! " # $ % & ' ( ) * + , - . / ...

        # Cache for get_corpus_embeddings(): the mean-vector computation
        # is done once and reused on later calls.
        self._corpus_embeddings = None

        # ---- Step 1: preprocess every abstract ------------------------
        # Word2Vec consumes a list of token lists, so preprocessing must
        # happen before training. The SAME function is later applied to
        # the query, guaranteeing corpus and query are treated identically.
        print(f"Preprocessing {len(self.abstracts)} abstracts...")
        self.tokenized_corpus = [self.preprocess(abstract) for abstract in self.abstracts]

        # ---- Step 2: obtain the word vectors --------------------------
        if use_pretrained:
            # Option A — pre-trained vectors (downloaded on first use).
            # These are trained on billions of general-English words, so
            # they cover everyday vocabulary far better than our corpus,
            # but they miss corpus-specific jargon and are a big download.
            print(f"Loading pre-trained vectors '{pretrained_model_name}' (large one-off download)...")
            self.model = None
            self.word_vectors = api.load(pretrained_model_name)
        else:
            # Option B — train from scratch on the abstracts themselves.
            # Skip-gram + a low min_count works well on a small, dense,
            # domain-specific corpus like research-paper abstracts.
            print(f"Training Word2Vec on the corpus (vector_size={vector_size}, epochs={epochs})...")
            self.model = Word2Vec(
                sentences=self.tokenized_corpus,  # the preprocessed token lists
                vector_size=vector_size,          # dimensionality of each word vector
                window=window,                    # context words considered left/right
                min_count=min_count,              # ignore very rare words
                sg=sg,                            # 1 = skip-gram, 0 = CBOW
                epochs=epochs,                    # training passes over the corpus
                workers=workers,                  # parallel worker threads
                seed=random_seed,                 # reproducibility
            )
            # KeyedVectors: the trained word -> vector lookup table.
            self.word_vectors = self.model.wv

        # Every embedding this class returns has this dimensionality,
        # whether it came from a trained or a pre-trained model.
        self.vector_size = self.word_vectors.vector_size

        print(f"Word2VecSearcher initialized with {len(self.abstracts)} abstracts.")
        print(f"Vocabulary size:     {len(self.word_vectors.index_to_key)}")
        print(f"Embedding dimension: {self.vector_size}")

    # --------------------------------------------------------------
    # Preprocessing
    # --------------------------------------------------------------

    @staticmethod
    def _wordnet_pos(treebank_tag):
        """
        Map a Penn-Treebank POS tag to the POS constant WordNet expects.

        The lemmatizer needs to know the part of speech to be accurate:
        without it, "learning" (verb) stays "learning" and "was" stays
        "wa". Defaults to NOUN, which is WordNet's own default.
        """
        if treebank_tag.startswith("J"):
            return wordnet.ADJ      # adjective
        if treebank_tag.startswith("V"):
            return wordnet.VERB     # verb
        if treebank_tag.startswith("R"):
            return wordnet.ADV      # adverb
        return wordnet.NOUN         # noun (default)

    def preprocess(self, text):
        """
        Turn a raw string into a clean list of lemmatized content words.

        Steps
        -----
        0. Basic cleaning : strip HTML tags, URLs and LaTeX-ish markers.
        1. Lowercasing    : "Neural" and "neural" become the same token.
        2. Tokenization   : split the sentence into a list of words.
        3. Filtering      : drop punctuation, digits, single characters
                            and standard English stop words.
        4. Lemmatization  : reduce inflected forms to their base form
                            ("networks" -> "network", "learned" -> "learn")
                            so all variants share one vector.

        Parameters
        ----------
        text : str
            Raw abstract or query text.

        Returns
        -------
        list of str
            The cleaned, lemmatized tokens.
        """
        # Guard against NaN / non-string input coming from a CSV column
        if not isinstance(text, str):
            return []

        # --- Step 0: basic cleaning ---------------------------------
        text = re.sub(r"<[^>]+>", " ", text)          # remove HTML tags: <br>, <p>, </div>
        text = re.sub(r"http\S+|www\.\S+", " ", text) # remove URLs
        text = re.sub(r"\s+", " ", text).strip()      # collapse repeated whitespace

        # --- Step 1: lowercase --------------------------------------
        text = text.lower()

        # --- Step 2: tokenize ---------------------------------------
        tokens = word_tokenize(text)

        # --- Step 3: remove punctuation, numbers and stop words -----
        cleaned_tokens = []
        for token in tokens:
            # Strip punctuation attached to a word ("state-of-the-art" -> "stateoftheart",
            # "model." -> "model"). Tokens that were pure punctuation become "".
            token = "".join(char for char in token if char not in self.punctuation)

            if not token.isalpha():        # drops "", "2021", "3d", stray symbols
                continue
            if len(token) < 3:             # drops noise like "et", "al", "e"
                continue
            if token in self.stop_words:   # drops "the", "we", "of", "which", ...
                continue

            cleaned_tokens.append(token)

        if not cleaned_tokens:
            return []

        # --- Step 4: POS-aware lemmatization ------------------------
        # POS-tag first so the lemmatizer knows whether "training" is a
        # noun or a verb, then reduce each token to its dictionary form.
        tagged_tokens = nltk.pos_tag(cleaned_tokens)
        lemmatized_tokens = [
            self.lemmatizer.lemmatize(token, self._wordnet_pos(tag))
            for token, tag in tagged_tokens
        ]

        return lemmatized_tokens

    # --------------------------------------------------------------
    # Document representation: Average Word Embedding
    # --------------------------------------------------------------

    def _average_word_vectors(self, tokens):
        """
        Average the Word2Vec vectors of `tokens` into a single vector.

        This is the Average Word Embedding strategy that turns word-level
        Word2Vec output into a document-level representation.

        Out-of-vocabulary tokens (words the model never saw, or words
        removed by `min_count`) have no vector and are simply skipped.
        A document with no in-vocabulary token at all falls back to a
        zero vector, which keeps the output array rectangular.

        Parameters
        ----------
        tokens : list of str
            Preprocessed tokens of one document or query.

        Returns
        -------
        numpy.ndarray
            A 1D array of shape (vector_size,), dtype float32.
        """
        # Collect a vector for every token the model actually knows
        vectors = [self.word_vectors[token] for token in tokens if token in self.word_vectors]

        # No known word -> return a zero vector rather than crashing
        if not vectors:
            return np.zeros(self.vector_size, dtype=np.float32)

        # The document vector is the element-wise mean of its word vectors
        return np.mean(vectors, axis=0).astype(np.float32)

    # --------------------------------------------------------------
    # Public API
    # --------------------------------------------------------------

    def get_corpus_embeddings(self):
        """
        Generate document embeddings for ALL abstracts.

        Pipeline:
            Preprocessed tokens → Word2Vec lookup → mean vector → NumPy array

        The result is cached, so repeated calls are free.

        Returns
        -------
        numpy.ndarray
            A 2D array of shape (n_papers, vector_size).
            Example: 727 abstracts with 300-dim vectors -> (727, 300).
        """
        # Return the cached matrix if it has already been computed
        if self._corpus_embeddings is not None:
            return self._corpus_embeddings

        print(f"Generating average-word-embedding vectors for {len(self.tokenized_corpus)} abstracts...")

        # One mean vector per abstract, stacked into a 2D matrix.
        # Row order matches `self.abstracts` exactly, so row i always
        # corresponds to abstract i across TF-IDF, Word2Vec and BERT.
        self._corpus_embeddings = np.vstack(
            [self._average_word_vectors(tokens) for tokens in self.tokenized_corpus]
        )

        print(f"Corpus embeddings generated!")
        print(f"Shape: {self._corpus_embeddings.shape}")

        return self._corpus_embeddings

    def get_query_embedding(self, query_string):
        """
        Generate an embedding for a single user query.

        The query goes through the EXACT SAME preprocessing and the SAME
        word-vector table as the abstracts, so the query vector and the
        document vectors live in the same vector space and are directly
        comparable.

        Pipeline:
            Query → preprocessing → Word2Vec lookup → mean vector → NumPy array

        Parameters
        ----------
        query_string : str
            The user's natural-language search query.

        Returns
        -------
        numpy.ndarray
            A 1D array of shape (vector_size,).
            A query whose words are all out-of-vocabulary returns a
            zero vector.
        """
        # Step 1: identical preprocessing to the corpus
        query_tokens = self.preprocess(query_string)

        # Step 2: average the vectors of the query's known words
        return self._average_word_vectors(query_tokens)

In [6]:
print("Word2VecSearcher class defined successfully!")

Word2VecSearcher class defined successfully!


---
## Cell 5 — Load the Dataset

Load the research-paper abstracts from `abstract_sentences.csv` (provided by the Scholar Inbox authors).

**Important:** the CSV contains multiple rows per paper (one per annotated sentence), so `drop_duplicates()` on the `abstract` column gives one entry per unique paper. It preserves the original order, which guarantees that **row *i* means the same paper** across the TF-IDF, Word2Vec and BERT implementations.

**For Google Colab:** upload `abstract_sentences.csv` to the runtime (or mount Drive) — the lookup below already tries the current folder and the parent folder, since this notebook lives in `notebooks/` while the CSV sits in the project root.


In [7]:
# ============================================================
# Cell 5 — Load the Dataset
# ============================================================
# The dataset "abstract_sentences.csv" is provided by the
# Scholar Inbox authors. Each row contains an abstract with
# sentence-level annotations. The same abstract appears
# multiple times, so we use drop_duplicates() to get one
# entry per unique paper.
#
# drop_duplicates() preserves the original order, ensuring
# consistency across TF-IDF, Word2Vec, and BERT.
# ============================================================

import os
import pandas as pd

# This notebook lives in notebooks/ while the CSV lives in the
# project root, so try both locations (and Colab's upload folder).
CSV_CANDIDATES = [
    "abstract_sentences.csv",
    "../abstract_sentences.csv",
    "/content/abstract_sentences.csv",
]
csv_path = next((p for p in CSV_CANDIDATES if os.path.exists(p)), None)
if csv_path is None:
    raise FileNotFoundError(
        "abstract_sentences.csv not found. Place it in the project root "
        "(or upload it to the Colab runtime)."
    )

# 1. Load the dataset provided by the Scholar Inbox authors
df = pd.read_csv(csv_path)

# 2. Extract the 'abstract' column and drop duplicates
# drop_duplicates() guarantees that the order of papers stays EXACTLY the same
# for TF-IDF, Word2Vec, and BERT.
unique_abstracts = df['abstract'].drop_duplicates().dropna().tolist()

print(f"Loaded from: {csv_path}")
print(f"Rows in CSV (one per annotated sentence): {len(df)}")
print(f"Successfully loaded {len(unique_abstracts)} unique research papers!")

# ---------------------------------------------------------
# 'unique_abstracts' is now a standard Python list of strings.
# ---------------------------------------------------------

# Preview the first abstract (truncated for display)
print(f"\nExample abstract (first 300 characters):")
print(f"{unique_abstracts[0][:300]}...")

Loaded from: ../abstract_sentences.csv
Rows in CSV (one per annotated sentence): 2538
Successfully loaded 727 unique research papers!

Example abstract (first 300 characters):
We present an efficient method for joint optimization of topology, materials and lighting from multi-view image observations. Unlike recent multi-view reconstruction approaches, which typically produce entangled 3D representations encoded in neural networks, we output triangle meshes with spatially-...


---
## Cell 6 — Preprocessing Walk-through

Before training anything, let's see exactly what `preprocess()` does to a real abstract. This is the step that separates the Word2Vec arm from the BERT arm of the study: stop words, punctuation, numbers and inflection are all stripped, leaving only lemmatized content words.


In [8]:
# ============================================================
# Cell 6 — Preprocessing Walk-through
# ============================================================
# A throw-away instance on a tiny slice, just to expose the
# preprocess() method (no training happens here — we only need
# the tokenizer/stop-word/lemmatizer setup).
# ============================================================

demo_preprocessor = Word2VecSearcher(unique_abstracts[:1], min_count=1, epochs=1)

sample_text = unique_abstracts[0]
sample_tokens = demo_preprocessor.preprocess(sample_text)

print("=" * 70)
print("BEFORE — raw abstract (first 400 characters)")
print("=" * 70)
print(sample_text[:400], "...")

print()
print("=" * 70)
print("AFTER — preprocessed tokens (first 40)")
print("=" * 70)
print(sample_tokens[:40])

print()
print(f"Raw word count:          {len(sample_text.split())}")
print(f"Tokens after cleaning:   {len(sample_tokens)}")
print(f"Reduction:               {100 * (1 - len(sample_tokens) / len(sample_text.split())):.1f}% of tokens removed")
print()
print("Notice: lowercased, punctuation/numbers gone, stop words gone,")
print("        and plurals/inflections reduced to their base form.")

Preprocessing 1 abstracts...


Training Word2Vec on the corpus (vector_size=300, epochs=1)...
Word2VecSearcher initialized with 1 abstracts.
Vocabulary size:     80
Embedding dimension: 300
BEFORE — raw abstract (first 400 characters)
We present an efficient method for joint optimization of topology, materials and lighting from multi-view image observations. Unlike recent multi-view reconstruction approaches, which typically produce entangled 3D representations encoded in neural networks, we output triangle meshes with spatially-varying materials and environment lighting that can be deployed in any traditional graphics engine u ...

AFTER — preprocessed tokens (first 40)
['present', 'efficient', 'method', 'joint', 'optimization', 'topology', 'material', 'light', 'multiview', 'image', 'observation', 'unlike', 'recent', 'multiview', 'reconstruction', 'approach', 'typically', 'produce', 'entangled', 'representation', 'encode', 'neural', 'network', 'output', 'triangle', 'mesh', 'spatiallyvarying', 'material', 'environme

---
## Cell 7 — Initialize the `Word2VecSearcher` (trains the model)

Constructing the searcher preprocesses all abstracts and then trains Word2Vec on them.

**Why train from scratch rather than load Google News vectors?** Our corpus is highly domain-specific (computer-vision and ML abstracts). Terms like *"multi-view"*, *"differentiable"*, *"transformer"* or *"NeRF"* are either absent from general-English vectors or carry the wrong sense. Training on the corpus itself gives vectors tuned to exactly this vocabulary, costs seconds instead of a 1.6 GB download, and keeps the notebook runnable offline.

To use pre-trained vectors instead:

```python
searcher = Word2VecSearcher(unique_abstracts, use_pretrained=True)
```


In [9]:
# ============================================================
# Cell 7 — Initialize the Word2VecSearcher
# ============================================================
# This preprocesses every abstract and trains Word2Vec on the
# resulting token lists.
#
#   sg=1        -> skip-gram, which handles small corpora and
#                  rare technical terms better than CBOW
#   min_count=2 -> ignore words that appear only once
#   epochs=40   -> our corpus is small, so extra passes help
# ============================================================

searcher = Word2VecSearcher(
    unique_abstracts,
    use_pretrained=False,   # train on our own corpus
    vector_size=300,        # 300-dimensional word vectors
    window=5,               # context window
    min_count=2,            # skip words occurring only once
    epochs=40,              # training passes
    sg=1,                   # skip-gram
    random_seed=42,         # reproducible
)

Preprocessing 727 abstracts...


Training Word2Vec on the corpus (vector_size=300, epochs=40)...


Word2VecSearcher initialized with 727 abstracts.
Vocabulary size:     3904
Embedding dimension: 300


---
## Cell 8 — Generate Corpus Embeddings

Each abstract's preprocessed tokens are looked up in the trained model and averaged into a single vector.

Expected output: a 2D NumPy array with **one row per unique abstract** and **300 columns** (the vector size).


In [10]:
# ============================================================
# Cell 8 — Generate Corpus Embeddings
# ============================================================

corpus_embeddings = searcher.get_corpus_embeddings()

print(f"\nCorpus embeddings shape: {corpus_embeddings.shape}")
print(f"  -> {corpus_embeddings.shape[0]} documents, each a {corpus_embeddings.shape[1]}-dimensional vector")

Generating average-word-embedding vectors for 727 abstracts...
Corpus embeddings generated!
Shape: (727, 300)

Corpus embeddings shape: (727, 300)
  -> 727 documents, each a 300-dimensional vector


---
## Cell 9 — Generate a Query Embedding

The query passes through the **same** `preprocess()` and the **same** word-vector table as the abstracts, so both live in the same 300-dimensional space and are directly comparable.


In [11]:
# ============================================================
# Cell 9 — Generate Query Embedding
# ============================================================

query = "neural network for 3D scene reconstruction from images"

query_embedding = searcher.get_query_embedding(query)

print(f"Query:                 '{query}'")
print(f"Preprocessed tokens:   {searcher.preprocess(query)}")
print(f"Query embedding shape: {query_embedding.shape}")

Query:                 'neural network for 3D scene reconstruction from images'
Preprocessed tokens:   ['neural', 'network', 'scene', 'reconstruction', 'image']
Query embedding shape: (300,)


---
## Cell 10 — Inspect the Embeddings

Verify the generated embeddings are well-formed and that corpus and query share one vector space. **No similarity calculation or ranking happens here** — that is the deliverable boundary.


In [12]:
# ============================================================
# Cell 10 — Inspect Embeddings
# ============================================================

print("=" * 60)
print("CORPUS EMBEDDINGS")
print("=" * 60)
print(f"Type:            {type(corpus_embeddings)}")
print(f"Data type:       {corpus_embeddings.dtype}")
print(f"Shape:           {corpus_embeddings.shape}")
print(f"  -> {corpus_embeddings.shape[0]} documents, each represented as a {corpus_embeddings.shape[1]}-dimensional vector")
print(f"\nFirst document embedding (first 10 values):")
print(f"  {corpus_embeddings[0][:10]}")
print(f"\nEmbedding statistics:")
print(f"  Min:  {corpus_embeddings.min():.6f}")
print(f"  Max:  {corpus_embeddings.max():.6f}")
print(f"  Mean: {corpus_embeddings.mean():.6f}")
print(f"  Std:  {corpus_embeddings.std():.6f}")

# A document whose words were ALL out-of-vocabulary would be an all-zero row
zero_rows = int((corpus_embeddings == 0).all(axis=1).sum())
print(f"\nAll-zero (empty/OOV) document vectors: {zero_rows}")

print()
print("=" * 60)
print("QUERY EMBEDDING")
print("=" * 60)
print(f"Type:            {type(query_embedding)}")
print(f"Data type:       {query_embedding.dtype}")
print(f"Shape:           {query_embedding.shape}")
print(f"\nQuery embedding (first 10 values):")
print(f"  {query_embedding[:10]}")

print()
print("=" * 60)
print("VERIFICATION")
print("=" * 60)
print(f"Same dimension: {corpus_embeddings.shape[1] == query_embedding.shape[0]} "
      f"({corpus_embeddings.shape[1]} == {query_embedding.shape[0]})")
print(f"\n-> Both embeddings exist in the same {corpus_embeddings.shape[1]}-dimensional vector space.")
print(f"-> They can be compared using cosine similarity (handled by a separate component).")

CORPUS EMBEDDINGS
Type:            <class 'numpy.ndarray'>
Data type:       float32
Shape:           (727, 300)
  -> 727 documents, each represented as a 300-dimensional vector

First document embedding (first 10 values):
  [ 0.09988365  0.00379684 -0.11048732 -0.14338775 -0.13752836  0.01719118
  0.14144064  0.02559498 -0.04436432  0.11991589]

Embedding statistics:
  Min:  -0.492313
  Max:  0.461440
  Mean: -0.004961
  Std:  0.110771

All-zero (empty/OOV) document vectors: 0

QUERY EMBEDDING
Type:            <class 'numpy.ndarray'>
Data type:       float32
Shape:           (300,)

Query embedding (first 10 values):
  [-0.11683971  0.20686129 -0.11853942  0.04569831 -0.13024428 -0.01026744
  0.21296719  0.01007218  0.01620711  0.10204996]

VERIFICATION
Same dimension: True (300 == 300)

-> Both embeddings exist in the same 300-dimensional vector space.
-> They can be compared using cosine similarity (handled by a separate component).


---
# Demo Run — Semantic Search over the Dataset

Everything above is the deliverable. The cells below are a **demonstration** that the embeddings actually retrieve sensible papers.

The ranking logic here (cosine similarity + `argsort`) is intentionally written **outside** the `Word2VecSearcher` class, since the comparison/ranking component belongs to a different part of the case study.

---
## Cell 11 — A Small Search Helper (demo only)

Cosine similarity measures the **angle** between two vectors, ignoring their magnitude — which is what we want, since a long abstract and a three-word query should still be comparable.


In [13]:
# ============================================================
# Cell 11 — Search Helper (DEMO ONLY — not part of the class)
# ============================================================

from sklearn.metrics.pairwise import cosine_similarity


def search(query_string, top_k=5, preview_chars=260):
    """Rank abstracts by cosine similarity to the query and print the top-k."""
    # 1. Embed the query with the SAME searcher (same vector space)
    q_vec = searcher.get_query_embedding(query_string).reshape(1, -1)

    # 2. Cosine similarity against every document vector
    scores = cosine_similarity(q_vec, corpus_embeddings)[0]

    # 3. Take the top-k highest scores (descending)
    top_indices = np.argsort(scores)[::-1][:top_k]

    print("=" * 78)
    print(f"QUERY: {query_string}")
    print(f"Preprocessed tokens: {searcher.preprocess(query_string)}")
    print("=" * 78)
    for rank, idx in enumerate(top_indices, start=1):
        print(f"\n[{rank}] score = {scores[idx]:.4f}   (paper index {idx})")
        print(f"    {unique_abstracts[idx][:preview_chars].strip()}...")
    print()
    return top_indices, scores


print("Search helper ready!")

Search helper ready!


---
## Cell 12 — Run the Searches

Three queries, each deliberately phrased in **different words than the abstracts use**, to show that matching happens on meaning rather than on literal term overlap.


In [14]:
# ============================================================
# Cell 12 — Demo Searches
# ============================================================

demo_queries = [
    "reconstructing 3D shapes and lighting from photographs",
    "self-supervised pretraining of language models",
    "detecting objects in video in real time",
]

for q in demo_queries:
    search(q, top_k=3)
    print()

QUERY: reconstructing 3D shapes and lighting from photographs
Preprocessed tokens: ['reconstruct', 'shape', 'light', 'photograph']

[1] score = 0.7593   (paper index 658)
    We present a method to edit complex indoor lighting from a single image with its predicted depth and light source segmentation masks. This is an extremely challenging problem that requires modeling complex light transport, and disentangling HDR lighting from m...

[2] score = 0.7471   (paper index 601)
    A light stage uses a series of calibrated cameras and lights to capture a subject's facial appearance under varying illumination and viewpoint. This captured information is crucial for facial reconstruction and relighting. Unfortunately, light stages are...

[3] score = 0.7234   (paper index 534)
    We present a differentiable rendering framework for material and lighting estimation from multi-view images and a reconstructed geometry. In the framework, we represent scene lightings as the Neural Incident Light F

---
## Cell 13 — What Did the Model Actually Learn?

A quick look at the **word level** makes the "static semantic" idea concrete: these neighbours were never told to be related — the model inferred it purely from which words appear in similar contexts across the abstracts.


In [15]:
# ============================================================
# Cell 13 — Nearest Neighbours in the Learned Vector Space
# ============================================================

probe_words = ["image", "network", "training", "segmentation", "text"]

for word in probe_words:
    if word in searcher.word_vectors:
        neighbours = searcher.word_vectors.most_similar(word, topn=6)
        formatted = ", ".join(f"{w} ({s:.2f})" for w, s in neighbours)
        print(f"{word:<14} -> {formatted}")
    else:
        print(f"{word:<14} -> (not in vocabulary)")

print()
print("These relationships were learned from co-occurrence alone —")
print("no dictionary, no labels, no supervision.")

image          -> multiviewconsistent (0.45), renderready (0.40), inherits (0.40), reprojected (0.39), photograph (0.39), giraffe (0.38)
network        -> neural (0.46), convolutional (0.40), encoderdecoder (0.40), feasibility (0.39), explicitimplicit (0.38), radiosity (0.37)
training       -> cgans (0.40), randomly (0.38), unlabelled (0.38), ebms (0.37), later (0.36), paired (0.36)
segmentation   -> colorbased (0.52), singlemodal (0.49), primary (0.45), leaf (0.45), ggns (0.44), swsss (0.44)
text           -> prompt (0.46), textimage (0.44), ape (0.43), chemical (0.43), tagalog (0.43), asr (0.41)

These relationships were learned from co-occurrence alone —
no dictionary, no labels, no supervision.


---
## Summary

| | |
|---|---|
| **Model** | Word2Vec (skip-gram), trained on the corpus |
| **Preprocessing** | lowercase → tokenize → remove punctuation/numbers/stop words → POS-aware lemmatize |
| **Document strategy** | Average Word Embedding (mean of word vectors) |
| **Corpus output** | `get_corpus_embeddings()` → 2D array `(n_papers, 300)` |
| **Query output** | `get_query_embedding(q)` → 1D array `(300,)` |

**Strengths vs TF-IDF:** matches meaning, not literal words — a query and an abstract can score highly with zero shared vocabulary.

**Limitations (what the BERT arm addresses):**
- **Static vectors** — one vector per word, so polysemy ("bank", "transformer") is unresolved.
- **Averaging discards word order** — *"model predicts the image"* and *"image predicts the model"* produce identical document vectors.
- **Frequent generic words dilute the mean** — a long abstract's vector drifts toward the corpus average (a TF-IDF-weighted mean is the usual mitigation).
- **Out-of-vocabulary words are simply skipped** — a query made entirely of unseen terms yields a zero vector.
